In [1]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import matplotlib.pyplot as plt

Mounted at /content/drive


In [2]:
df_merged = pd.read_csv("/content/drive/MyDrive/CRIS/df_merged_full.csv")
df_merged.head()

,rating,title,text,images_x,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase,...,average_rating,rating_number,features,description,price,images_y,videos,store,categories,details
0,3.0,Battery life sucks,Battery lasted 2 months ...sucks,[],B085L34V8J,B085L34V8J,AESGKFNIDELMTPT7OLS67JA545JA,1603999669246,0,True,...,4.1,1091,[],[],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[{'title': 'Gorsun Wireless Bluetooth Sports E...,gorsun,"['Electronics', 'Headphones, Earbuds & Accesso...",{'Package Dimensions': '7.32 x 5.94 x 1.26 inc...
1,1.0,JUNK!!!,These are absolute junk. Not at all the Bose ...,[],B01L7PWBRG,B0B62FJF1J,AEAU4B3HGL46SP3ARQZZ3B7PDA4Q,1489026041000,0,False,...,4.4,39355,['Note : If the size of the earbud tips does n...,[],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],Bose,"['Electronics', 'Headphones, Earbuds & Accesso...","{'Brand': 'Bose', 'Model Name': 'SoundSport', ..."
2,2.0,Poor connection,"Crappy sound quality, poor fit, volume control...",[],B07CX38CQ2,B07GC8NMB8,AE427MY65PZ6FNZ2IY6HCTIIQZEQ,1565129082806,0,True,...,4.0,872,['Bluetooth wireless allows you to stream high...,['You will receive 1 JBL endurance Sprint - Sp...,NaN,[{'thumb': 'https://m.media-amazon.com/images/...,"[{'title': 'Perfect earbuds for working out!',...",JBL,"['Electronics', 'Headphones, Earbuds & Accesso...",{'Product Dimensions': '1.65 x 6.38 x 4.02 inc...
3,2.0,Poor connection,"Crappy sound quality, poor fit, volume control...",[],B07CX38CQ2,B07GC8NMB8,AE427MY65PZ6FNZ2IY6HCTIIQZEQ,1565129082806,0,True,...,4.0,872,['Bluetooth wireless allows you to stream high...,['You will receive 1 JBL endurance Sprint - Sp...,NaN,[{'thumb': 'https://m.media-amazon.com/images/...,"[{'title': 'Perfect earbuds for working out!',...",JBL,"['Electronics', 'Headphones, Earbuds & Accesso...",{'Product Dimensions': '1.65 x 6.38 x 4.02 inc...
4,2.0,Poor connection,"Crappy sound quality, poor fit, volume control...",[],B07CX38CQ2,B07GC8NMB8,AE427MY65PZ6FNZ2IY6HCTIIQZEQ,1565129082806,0,True,...,4.0,872,['Bluetooth wireless allows you to stream high...,['You will receive 1 JBL endurance Sprint - Sp...,NaN,[{'thumb': 'https://m.media-amazon.com/images/...,"[{'title': 'Perfect earbuds for working out!',...",JBL,"['Electronics', 'Headphones, Earbuds & Accesso...",{'Product Dimensions': '1.65 x 6.38 x 4.02 inc...


In [3]:
df_merged["store"].value_counts().head(20)

,count
store,
Sony,938
phaiser,621
Skullcandy,362
Kinivo,346
Bose,322
Audio-Technica,308
Amazon Renewed,304
Jaybird,300
Logitech G,294


In [4]:
target_brands = [
    "Sony",
    "Bose",
    "Skullcandy",
    "Audio-Technica",
    "Sennheiser Consumer Audio",
    "JVC",
    "LG",
    "Anker",
    "Soundcore",
    "SoundPEATS"
]

In [5]:
brand_df = df_merged[df_merged["store"].isin(target_brands)].copy()
brand_df.shape

(2901, 32)

In [6]:
brand_df["store"] = brand_df["store"].replace({
    "Sennheiser Consumer Audio": "Sennheiser"
})

In [7]:
brand_volume = brand_df["store"].value_counts()
brand_volume

,count
store,
Sony,938
Skullcandy,362
Bose,322
Audio-Technica,308
Sennheiser,227
Anker,178
SoundPEATS,172
LG,139
Soundcore,132


In [8]:
brand_aspect = brand_df.groupby(["store", "aspects"])["rating"].mean().reset_index()

pivot = brand_aspect.pivot(index="store", columns="aspects", values="rating")

pivot

aspects,anc,battery,build,comfort,connectivity,microphone,price,sound
store,,,,,,,,
Anker,5.000000,4.384615,4.000000,4.175000,4.235294,3.800000,4.619048,4.146341
Audio-Technica,4.478261,4.833333,4.595745,4.486111,4.240000,4.555556,4.615385,4.609195
Bose,3.631579,3.674419,3.531250,3.760563,3.775000,3.250000,3.400000,3.913580
JVC,4.250000,3.812500,3.375000,3.769231,3.833333,4.000000,4.000000,3.851852
LG,4.000000,4.142857,3.750000,3.406250,4.000000,3.153846,4.166667,3.708333
Sennheiser,4.500000,4.150000,4.379310,4.300000,4.437500,4.058824,4.450000,4.491228
Skullcandy,3.611111,3.473684,3.019231,3.536232,3.444444,4.066667,3.857143,3.540541
Sony,3.889831,4.038961,3.825688,3.905263,3.873786,3.650000,3.756757,3.975845
SoundPEATS,4.000000,3.631579,3.500000,3.571429,3.545455,4.000000,4.294118,3.729730


In [9]:
pivot.idxmax()

,0
aspects,
anc,Anker
battery,Audio-Technica
build,Audio-Technica
comfort,Audio-Technica
connectivity,Sennheiser
microphone,Audio-Technica
price,Anker
sound,Audio-Technica


In [10]:
negative_df = brand_df[brand_df["rating"] <= 2]

neg_counts = negative_df.groupby(["store", "aspects"]).size().reset_index(name="count")

total_reviews = brand_df.groupby("store").size()

neg_counts["total_reviews"] = neg_counts["store"].map(total_reviews)
neg_counts["negative_rate"] = neg_counts["count"] / neg_counts["total_reviews"]

neg_pivot = neg_counts.pivot(index="store", columns="aspects", values="negative_rate").fillna(0)

neg_pivot

aspects,anc,battery,build,comfort,connectivity,microphone,price,sound
store,,,,,,,,
Anker,0.000000,0.005618,0.016854,0.028090,0.011236,0.011236,0.005618,0.028090
Audio-Technica,0.003247,0.000000,0.006494,0.012987,0.009740,0.000000,0.006494,0.003247
Bose,0.018634,0.037267,0.027950,0.055901,0.034161,0.018634,0.021739,0.049689
JVC,0.000000,0.016260,0.032520,0.032520,0.016260,0.000000,0.000000,0.032520
LG,0.007194,0.021583,0.035971,0.079137,0.021583,0.043165,0.007194,0.043165
Sennheiser,0.004405,0.013216,0.004405,0.026432,0.004405,0.008811,0.004405,0.008811
Skullcandy,0.005525,0.024862,0.055249,0.049724,0.035912,0.002762,0.019337,0.046961
Sony,0.019190,0.013859,0.019190,0.034115,0.018124,0.011727,0.014925,0.035181
SoundPEATS,0.005814,0.023256,0.034884,0.052326,0.034884,0.005814,0.005814,0.046512


In [11]:
neg_pivot.idxmax(axis=1)

,0
store,
Anker,comfort
Audio-Technica,comfort
Bose,comfort
JVC,build
LG,comfort
Sennheiser,comfort
Skullcandy,build
Sony,sound
SoundPEATS,comfort


In [12]:
summary = brand_df.groupby("store").agg(
    total_reviews=("rating", "count"),
    avg_rating=("rating", "mean"),
    avg_price=("price", "mean"),
    avg_product_rating=("average_rating", "mean"),
    avg_helpful_vote=("helpful_vote", "mean")   # ADD THIS
)

summary

,total_reviews,avg_rating,avg_price,avg_product_rating,avg_helpful_vote
store,,,,,
Anker,178,4.230337,NaN,3.955056,0.960674
Audio-Technica,308,4.542208,199.000000,4.383766,3.448052
Bose,322,3.711180,NaN,4.381056,0.692547
JVC,123,3.796748,18.549000,3.785366,1.430894
LG,139,3.726619,104.990000,4.093525,0.366906
Sennheiser,227,4.356828,161.062500,4.196916,6.308370
Skullcandy,362,3.505525,33.024087,4.227072,1.535912
Sony,938,3.889126,219.273036,4.294989,2.814499
SoundPEATS,172,3.720930,NaN,3.661628,0.401163


In [13]:
pivot
neg_pivot
summary

,total_reviews,avg_rating,avg_price,avg_product_rating,avg_helpful_vote
store,,,,,
Anker,178,4.230337,NaN,3.955056,0.960674
Audio-Technica,308,4.542208,199.000000,4.383766,3.448052
Bose,322,3.711180,NaN,4.381056,0.692547
JVC,123,3.796748,18.549000,3.785366,1.430894
LG,139,3.726619,104.990000,4.093525,0.366906
Sennheiser,227,4.356828,161.062500,4.196916,6.308370
Skullcandy,362,3.505525,33.024087,4.227072,1.535912
Sony,938,3.889126,219.273036,4.294989,2.814499
SoundPEATS,172,3.720930,NaN,3.661628,0.401163


In [14]:
pivot
neg_pivot

aspects,anc,battery,build,comfort,connectivity,microphone,price,sound
store,,,,,,,,
Anker,0.000000,0.005618,0.016854,0.028090,0.011236,0.011236,0.005618,0.028090
Audio-Technica,0.003247,0.000000,0.006494,0.012987,0.009740,0.000000,0.006494,0.003247
Bose,0.018634,0.037267,0.027950,0.055901,0.034161,0.018634,0.021739,0.049689
JVC,0.000000,0.016260,0.032520,0.032520,0.016260,0.000000,0.000000,0.032520
LG,0.007194,0.021583,0.035971,0.079137,0.021583,0.043165,0.007194,0.043165
Sennheiser,0.004405,0.013216,0.004405,0.026432,0.004405,0.008811,0.004405,0.008811
Skullcandy,0.005525,0.024862,0.055249,0.049724,0.035912,0.002762,0.019337,0.046961
Sony,0.019190,0.013859,0.019190,0.034115,0.018124,0.011727,0.014925,0.035181
SoundPEATS,0.005814,0.023256,0.034884,0.052326,0.034884,0.005814,0.005814,0.046512


In [15]:
pivot.to_csv("/content/drive/MyDrive/CRIS/brand_aspect_rating.csv")
neg_pivot.to_csv("/content/drive/MyDrive/CRIS/brand_negative_rate.csv")
summary.to_csv("/content/drive/MyDrive/CRIS/competitor_summary_table.csv")
brand_df.to_csv("/content/drive/MyDrive/CRIS/brand_df.csv", index=False)
negative_df.to_csv("/content/drive/MyDrive/CRIS/negative_brand_reviews.csv", index=False)

In [16]:
print(summary.columns)

Index(['total_reviews', 'avg_rating', 'avg_price', 'avg_product_rating',
       'avg_helpful_vote'],
      dtype='object')
